# Server Dataset Allocation — Algorithm Comparison

Тестирование двух алгоритмов распределения серверного датасета по клиентам.

**Цель:** найти алгоритм, который при ограниченном серверном бюджете реально снижает
межклиентскую гетерогенность (`mean_pairwise_JS`).

| Алгоритм | Целевое распределение | Логика |
|---|---|---|
| **exact-uniform** (текущий) | Q_i → 1/K для каждого клиента | s_i[k] = (n_i+S_i)/K − n_i·P_i[k] |
| **prop-deficit** (новый) | пропорционально дефициту | s_i[k] = per_class · deficit_ik / Σ_j deficit_jk |

**Матрица экспериментов:**
- α ∈ {0.1, 0.2, ..., 0.9} (Dirichlet, 6 клиентов, CIFAR-10 10 классов)
- server_size ∈ {0, 1000, 3000, 5000, 10000, 15000, 20000}
- режим exclusive (sum_i s_i[k] ≤ per_class)
- 20 случайных seed для усреднения

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import itertools
import warnings
warnings.filterwarnings('ignore')

np.set_printoptions(precision=4, suppress=True)
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

# ── Параметры CIFAR-10 / FL ────────────────────────────────────────────────
N_CLIENTS       = 6
N_CLASSES       = 10
TRAIN_SIZE      = 50_000   # CIFAR-10 train
PER_CLASS_TOTAL = TRAIN_SIZE // N_CLASSES   # 5000
N_SEEDS         = 20

ALPHAS       = [round(a, 1) for a in np.arange(0.1, 1.0, 0.1)]
SERVER_SIZES = [0, 1000, 3000, 5000, 10000, 15000, 20000]

CLASS_NAMES = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']

print(f'Alphas       : {ALPHAS}')
print(f'Server sizes : {SERVER_SIZES}')
print(f'Seeds        : {N_SEEDS}')
print(f'Experiments  : {len(ALPHAS) * len(SERVER_SIZES) * N_SEEDS:,}')

## 1. Математические утилиты

In [ ]:
def js_divergence(p: np.ndarray, q: np.ndarray) -> float:
    """Jensen-Shannon divergence (base e). Symmetric, 0 ≤ JS ≤ ln(2)."""
    p = np.asarray(p, float); q = np.asarray(q, float)
    p = p / p.sum(); q = q / q.sum()
    m = 0.5 * (p + q)
    with np.errstate(divide='ignore', invalid='ignore'):
        js = (0.5 * np.where(p > 0, p * np.log(p / m), 0.0).sum() +
              0.5 * np.where(q > 0, q * np.log(q / m), 0.0).sum())
    return float(np.clip(js, 0.0, None))


def mean_pairwise_js(dists: list) -> float:
    """Среднее JS по всем парам клиентов."""
    n = len(dists)
    if n < 2:
        return 0.0
    return float(np.mean([js_divergence(dists[i], dists[j])
                           for i, j in itertools.combinations(range(n), 2)]))


def entropy_norm(dist: np.ndarray) -> float:
    """Нормализованная энтропия ∈ [0,1]. 1=IID, 0=один класс."""
    p = np.asarray(dist, float); p = p / p.sum()
    with np.errstate(divide='ignore'):
        h = -np.where(p > 0, p * np.log(p), 0.0).sum()
    return float(h / np.log(len(p)))

## 2. Партиционирование

In [ ]:
def dirichlet_partition(per_class_avail: int, n_clients: int, alpha: float,
                         n_classes: int, rng: np.random.Generator) -> np.ndarray:
    """
    Partitions `per_class_avail` samples per class across n_clients using Dirichlet(alpha).
    Returns counts[n_clients, n_classes].
    """
    counts = np.zeros((n_clients, n_classes), dtype=int)
    for k in range(n_classes):
        props = rng.dirichlet(np.full(n_clients, alpha))
        raw   = props * per_class_avail
        fl    = np.floor(raw).astype(int)
        rem   = per_class_avail - fl.sum()
        idx   = np.argsort(-(raw - fl))[:rem]
        fl[idx] += 1
        counts[:, k] = fl
    return counts


def build_profiles(counts: np.ndarray) -> list:
    """
    counts[n_clients, n_classes] → list of dicts {pid, n, P}.
    P = normalized distribution.
    """
    profiles = []
    for i, c in enumerate(counts):
        n = int(c.sum())
        P = c / n if n > 0 else np.zeros(c.shape)
        profiles.append({'pid': i, 'n': n, 'counts': c.copy(), 'P': P})
    return profiles


# Быстрая проверка
rng_test = np.random.default_rng(0)
c = dirichlet_partition(4500, 6, 0.3, 10, rng_test)
p = build_profiles(c)
print('Partition α=0.3, server extracted 5000 samples (500/class):')
print(f'  Client sample counts : {[x["n"] for x in p]}')
print(f'  Sum                  : {sum(x["n"] for x in p):,} (expect 45000)')
print(f'  mean pairwise JS     : {mean_pairwise_js([x["P"] for x in p]):.4f}')

## 3. Алгоритмы распределения серверного датасета

In [ ]:
# ── Общие утилиты ─────────────────────────────────────────────────────────

def apply_exclusive(schedule: np.ndarray, per_class: int) -> np.ndarray:
    """Exclusive: sum_i(s_i[k]) ≤ per_class для каждого класса k."""
    result = schedule.copy()
    for k in range(schedule.shape[1]):
        total = result[:, k].sum()
        if total > per_class:
            result[:, k] *= per_class / total
    return result


def effective_distributions(profiles: list, schedule: np.ndarray) -> list:
    """
    Q_i[k] = (n_i * P_i[k] + s_i[k]) / (n_i + sum_k s_i[k])
    Возвращает список numpy-массивов Q_i.
    """
    Qs = []
    for i, prof in enumerate(profiles):
        n_i = prof['n']
        s_i = schedule[i]
        n_added = s_i.sum()
        if n_i + n_added > 0:
            Q = (n_i * prof['P'] + s_i) / (n_i + n_added)
        else:
            Q = prof['P'].copy()
        Qs.append(Q)
    return Qs


# ── Алгоритм 1: Exact-Uniform (текущий в prod) ────────────────────────────

def algo_exact_uniform(profiles: list, per_class: int, n_classes: int) -> np.ndarray:
    """
    Для каждого клиента i: s_i[k] так чтобы Q_i → uniform (1/K).
    S_i = n_i * (D/K - U) / (1 - D/K)
    s_i[k] = max(0, (n_i + S_i)/K - n_i * P_i[k])
    Затем exclusive: sum_i s_i[k] ≤ per_class.
    """
    K = n_classes
    schedule = np.zeros((len(profiles), K), dtype=float)
    for i, prof in enumerate(profiles):
        n_i = prof['n']
        P   = prof['P']
        under = P < 1/K
        D = under.sum()
        U = P[under].sum()
        if D == 0 or D == K:
            continue
        denom = 1 - D/K
        if denom <= 0:
            continue
        S_i = n_i * (D/K - U) / denom
        for k in range(K):
            schedule[i, k] = max(0.0, (n_i + S_i)/K - n_i * P[k])
    return apply_exclusive(schedule, per_class)


# ── Алгоритм 2: Proportional-Deficit (новый) ──────────────────────────────

def algo_prop_deficit(profiles: list, per_class: int, n_classes: int) -> np.ndarray:
    """
    Для каждого класса k: распределить per_class пропорционально дефициту.
    deficit_i[k] = max(0, 1/K - P_i[k])
    s_i[k] = per_class * deficit_i[k] / sum_j deficit_j[k]
    Exclusive выполняется автоматически (sum_i = per_class по построению).
    """
    K = n_classes
    n = len(profiles)
    schedule = np.zeros((n, K), dtype=float)
    for k in range(K):
        deficits = np.array([max(0.0, 1/K - prof['P'][k]) for prof in profiles])
        total = deficits.sum()
        if total > 0:
            schedule[:, k] = per_class * deficits / total
    return schedule


# ── Алгоритм 3: Prop-Deficit с взвешиванием по n_i ────────────────────────

def algo_prop_deficit_weighted(profiles: list, per_class: int, n_classes: int) -> np.ndarray:
    """
    Как prop-deficit, но дефицит взвешен по размеру клиента:
    weight_i[k] = n_i * max(0, 1/K - P_i[k])
    Интерпретация: пропорционально АБСОЛЮТНОМУ числу недостающих сэмплов.
    """
    K = n_classes
    n = len(profiles)
    schedule = np.zeros((n, K), dtype=float)
    for k in range(K):
        weights = np.array([prof['n'] * max(0.0, 1/K - prof['P'][k]) for prof in profiles])
        total = weights.sum()
        if total > 0:
            schedule[:, k] = per_class * weights / total
    return schedule


print('Алгоритмы определены:')
print('  1. algo_exact_uniform        — текущий в prod')
print('  2. algo_prop_deficit         — новый (пропорционально дефициту)')
print('  3. algo_prop_deficit_weighted — новый (взвешен по n_i)')

## 4. Sanity check на одном примере

In [ ]:
def print_algo_comparison(profiles, per_class, n_classes, label=''):
    if label:
        print(f'\n{'═'*70}')
        print(f'  {label}')
        print(f'{'═'*70}')

    js_before = mean_pairwise_js([p['P'] for p in profiles])
    print(f'  JS до : {js_before:.4f}')
    print(f'  Клиенты: {[p["n"] for p in profiles]}')
    print()

    algos = [
        ('exact-uniform',   algo_exact_uniform),
        ('prop-deficit',    algo_prop_deficit),
        ('prop-def-weight', algo_prop_deficit_weighted),
    ]
    results = []
    for name, fn in algos:
        sched = fn(profiles, per_class, n_classes)
        Qs    = effective_distributions(profiles, sched)
        js_after = mean_pairwise_js(Qs)
        total_used = sched.sum()
        budget_total = per_class * n_classes
        reduction = (js_before - js_after) / js_before * 100
        results.append({
            'Алгоритм': name,
            'JS до':    f'{js_before:.4f}',
            'JS после': f'{js_after:.4f}',
            'Δ JS':     f'{js_after - js_before:+.4f}',
            'Ред., %':  f'{reduction:+.1f}%',
            'Бюджет':   f'{total_used:.0f}/{budget_total}',
        })

    df = pd.DataFrame(results)
    print(df.to_string(index=False))
    return results


# Тест: α=0.3, server=3000
rng_test = np.random.default_rng(42)
server_per_class = 3000 // N_CLASSES   # 300
counts = dirichlet_partition(PER_CLASS_TOTAL - server_per_class, N_CLIENTS, 0.3, N_CLASSES, rng_test)
profiles_test = build_profiles(counts)
print_algo_comparison(profiles_test, server_per_class, N_CLASSES, 'α=0.3, server=3000')

# Тест: α=0.3, server=15000
rng_test2 = np.random.default_rng(42)
server_per_class_big = 15000 // N_CLASSES   # 1500
counts2 = dirichlet_partition(PER_CLASS_TOTAL - server_per_class_big, N_CLIENTS, 0.3, N_CLASSES, rng_test2)
profiles_test2 = build_profiles(counts2)
print_algo_comparison(profiles_test2, server_per_class_big, N_CLASSES, 'α=0.3, server=15000')

## 5. Полный sweep: α × server_size × seeds

In [ ]:
records = []

total = len(ALPHAS) * len(SERVER_SIZES) * N_SEEDS
done  = 0

for alpha in ALPHAS:
    for srv_size in SERVER_SIZES:
        js_b_list, js_eu_list, js_pd_list, js_pdw_list = [], [], [], []

        for seed in range(N_SEEDS):
            rng = np.random.default_rng(seed)

            per_class_srv  = srv_size // N_CLASSES
            per_class_avail = PER_CLASS_TOTAL - per_class_srv

            counts   = dirichlet_partition(per_class_avail, N_CLIENTS, alpha, N_CLASSES, rng)
            profiles = build_profiles(counts)

            Ps = [p['P'] for p in profiles]
            js_b = mean_pairwise_js(Ps)
            js_b_list.append(js_b)

            if srv_size == 0:
                js_eu_list.append(js_b)
                js_pd_list.append(js_b)
                js_pdw_list.append(js_b)
            else:
                sched_eu  = algo_exact_uniform(profiles, per_class_srv, N_CLASSES)
                sched_pd  = algo_prop_deficit(profiles, per_class_srv, N_CLASSES)
                sched_pdw = algo_prop_deficit_weighted(profiles, per_class_srv, N_CLASSES)

                js_eu_list.append(mean_pairwise_js(effective_distributions(profiles, sched_eu)))
                js_pd_list.append(mean_pairwise_js(effective_distributions(profiles, sched_pd)))
                js_pdw_list.append(mean_pairwise_js(effective_distributions(profiles, sched_pdw)))

            done += 1

        def red(before_list, after_list):
            vals = [(b - a) / b * 100 if b > 0 else 0
                    for b, a in zip(before_list, after_list)]
            return np.mean(vals), np.std(vals)

        eu_mean,  eu_std  = red(js_b_list, js_eu_list)
        pd_mean,  pd_std  = red(js_b_list, js_pd_list)
        pdw_mean, pdw_std = red(js_b_list, js_pdw_list)

        records.append({
            'alpha':        alpha,
            'server_size':  srv_size,
            'srv_pct':      round(srv_size / TRAIN_SIZE * 100, 1),
            'js_before':    np.mean(js_b_list),
            'js_eu':        np.mean(js_eu_list),
            'js_pd':        np.mean(js_pd_list),
            'js_pdw':       np.mean(js_pdw_list),
            'red_eu':       eu_mean,
            'red_pd':       pd_mean,
            'red_pdw':      pdw_mean,
            'std_eu':       eu_std,
            'std_pd':       pd_std,
            'std_pdw':      pdw_std,
        })

    print(f'  α={alpha:.1f}  done={done}/{total}')

df = pd.DataFrame(records)
print(f'\nГотово. Строк: {len(df)}')

## 6. Сводная таблица: JS-reduction (%) по алгоритмам

In [ ]:
def _box_table(headers, rows, title=None):
    widths = [len(str(h)) for h in headers]
    for row in rows:
        for i, v in enumerate(row):
            widths[i] = max(widths[i], len(str(v)))
    def _border(l, m, r):
        return l + m.join('─' * (w + 2) for w in widths) + r
    def _row(cells):
        return '│' + '│'.join(f' {str(v):<{widths[i]}} ' for i, v in enumerate(cells)) + '│'
    if title:
        print(f'\n{title}')
    print(_border('┌', '┬', '┐'))
    print(_row(headers))
    print(_border('├', '┼', '┤'))
    for i, row in enumerate(rows):
        print(_row(row))
        if i < len(rows) - 1:
            print(_border('├', '┼', '┤'))
    print(_border('└', '┴', '┘'))


# Таблица: строки=alpha, колонки=server_size, значения=reduction для каждого алгоритма
for algo, col, std_col in [
    ('exact-uniform (текущий)',   'red_eu',  'std_eu'),
    ('prop-deficit (новый)',      'red_pd',  'std_pd'),
    ('prop-deficit+weight',       'red_pdw', 'std_pdw'),
]:
    non_zero_sizes = [s for s in SERVER_SIZES if s > 0]
    headers = ['α \\ srv'] + [f'{s//1000}k' for s in non_zero_sizes]
    rows = []
    for alpha in ALPHAS:
        row = [f'{alpha:.1f}']
        for s in non_zero_sizes:
            r = df[(df['alpha'] == alpha) & (df['server_size'] == s)].iloc[0]
            val = r[col]
            std = r[std_col]
            # Отрицательные = JS вырос (плохо)
            sign = '+' if val >= 0 else ''
            row.append(f'{sign}{val:.1f}±{std:.1f}')
        rows.append(row)
    _box_table(headers, rows, title=f'JS Reduction % — {algo}')

print('\n  Положительные значения = JS снизился (хорошо).')
print('  Отрицательные = JS вырос (алгоритм мешает).')

## 7. Разница между алгоритмами: prop-deficit ЛУЧШЕ exact-uniform на X%?

In [ ]:
# Δ = red_pd - red_eu (положительное = prop-deficit лучше)
non_zero_sizes = [s for s in SERVER_SIZES if s > 0]
headers = ['α \\ srv'] + [f'{s//1000}k' for s in non_zero_sizes]
rows = []
for alpha in ALPHAS:
    row = [f'{alpha:.1f}']
    for s in non_zero_sizes:
        r = df[(df['alpha'] == alpha) & (df['server_size'] == s)].iloc[0]
        delta = r['red_pd'] - r['red_eu']
        sign = '+' if delta >= 0 else ''
        row.append(f'{sign}{delta:.1f}pp')
    rows.append(row)
_box_table(headers, rows, title='Δ = prop-deficit − exact-uniform  (в процентных пунктах JS-reduction)')
print('  Положительное: prop-deficit лучше.  Отрицательное: exact-uniform лучше.')

## 8. Визуализация

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

non_zero_sizes = [s for s in SERVER_SIZES if s > 0]

for ax, (algo, col, title) in zip(axes, [
    ('exact-uniform', 'red_eu',  'exact-uniform (текущий)'),
    ('prop-deficit',  'red_pd',  'prop-deficit (новый)'),
    ('prop-def-w',    'red_pdw', 'prop-deficit+weight'),
]):
    matrix = np.zeros((len(ALPHAS), len(non_zero_sizes)))
    for i, alpha in enumerate(ALPHAS):
        for j, s in enumerate(non_zero_sizes):
            r = df[(df['alpha'] == alpha) & (df['server_size'] == s)].iloc[0]
            matrix[i, j] = r[col]

    vmax = max(abs(matrix.min()), abs(matrix.max()))
    im = ax.imshow(matrix, cmap='RdYlGn', vmin=-vmax, vmax=vmax, aspect='auto')
    plt.colorbar(im, ax=ax, label='JS reduction %')
    ax.set_xticks(range(len(non_zero_sizes)))
    ax.set_xticklabels([f'{s//1000}k' for s in non_zero_sizes])
    ax.set_yticks(range(len(ALPHAS)))
    ax.set_yticklabels([f'{a:.1f}' for a in ALPHAS])
    ax.set_xlabel('server_size')
    ax.set_ylabel('alpha')
    ax.set_title(f'{title}\nJS Reduction %', fontsize=10)
    for i in range(len(ALPHAS)):
        for j in range(len(non_zero_sizes)):
            v = matrix[i, j]
            ax.text(j, i, f'{v:.0f}', ha='center', va='center', fontsize=7,
                    color='black' if abs(v) < vmax * 0.6 else 'white')

plt.suptitle('JS Reduction % — сравнение алгоритмов\n'
             'Зелёный = JS снизился, Красный = JS вырос', fontsize=11)
plt.tight_layout()
plt.savefig('server_algo_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# JS reduction vs server_size для разных alpha
fig, axes = plt.subplots(3, 3, figsize=(15, 12), sharey=False)
axes = axes.flatten()

for ax, alpha in zip(axes, ALPHAS):
    sub = df[df['alpha'] == alpha]
    sizes = sub['server_size'].tolist()
    pcts  = sub['srv_pct'].tolist()

    ax.axhline(0, color='gray', lw=0.8, ls='--')
    ax.plot(pcts, sub['red_eu'],  'o-', color='#e74c3c', label='exact-uniform',   lw=2)
    ax.plot(pcts, sub['red_pd'],  's-', color='#2ecc71', label='prop-deficit',    lw=2)
    ax.plot(pcts, sub['red_pdw'], '^-', color='#3498db', label='prop-def+weight', lw=2)

    ax.fill_between(pcts,
                    np.array(sub['red_eu']) - np.array(sub['std_eu']),
                    np.array(sub['red_eu']) + np.array(sub['std_eu']),
                    alpha=0.15, color='#e74c3c')
    ax.fill_between(pcts,
                    np.array(sub['red_pd']) - np.array(sub['std_pd']),
                    np.array(sub['red_pd']) + np.array(sub['std_pd']),
                    alpha=0.15, color='#2ecc71')

    js_b = sub[sub['server_size'] == 0]['js_before'].values[0]
    ax.set_title(f'α={alpha:.1f}  (JS_before={js_b:.3f})', fontsize=9)
    ax.set_xlabel('server_size (% train)', fontsize=8)
    ax.set_ylabel('JS reduction %', fontsize=8)
    ax.tick_params(labelsize=7)
    ax.grid(True, alpha=0.3)
    if alpha == ALPHAS[0]:
        ax.legend(fontsize=7)

plt.suptitle('JS Reduction % vs server_size по алгоритмам\n'
             'Выше = лучше. Ниже 0 = алгоритм вредит.', fontsize=11)
plt.tight_layout()
plt.savefig('server_algo_curves.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Абсолютный JS (не reduction) для server_size=3000 (наш текущий) и 15000
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, srv in [(axes[0], 3000), (axes[1], 15000)]:
    sub = df[df['server_size'] == srv]
    ax.plot(sub['alpha'], sub['js_before'], 'k--o', label='до (no server)', lw=2)
    ax.plot(sub['alpha'], sub['js_eu'],     'o-',   color='#e74c3c', label='exact-uniform', lw=2)
    ax.plot(sub['alpha'], sub['js_pd'],     's-',   color='#2ecc71', label='prop-deficit',  lw=2)
    ax.plot(sub['alpha'], sub['js_pdw'],    '^-',   color='#3498db', label='prop-def+weight', lw=2)
    ax.set_xlabel('alpha (Dirichlet)')
    ax.set_ylabel('mean pairwise JS')
    ax.set_title(f'server_size={srv:,} ({srv/TRAIN_SIZE*100:.0f}% train)')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_xticks([round(a, 1) for a in ALPHAS])

plt.suptitle('Абсолютный mean pairwise JS — до и после распределения серверного датасета', fontsize=11)
plt.tight_layout()
plt.savefig('server_algo_abs_js.png', dpi=120, bbox_inches='tight')
plt.show()

## 9. Минимальный server_size для >10% JS reduction

In [ ]:
threshold = 10.0  # % JS reduction

headers = ['alpha', 'JS_before', 'min_srv (exact-uniform)', 'min_srv (prop-deficit)', 'min_srv (prop-def+w)']
rows = []

for alpha in ALPHAS:
    sub = df[(df['alpha'] == alpha) & (df['server_size'] > 0)]
    js_b = df[(df['alpha'] == alpha) & (df['server_size'] == 0)]['js_before'].values[0]

    def min_srv(col):
        ok = sub[sub[col] >= threshold]
        if ok.empty:
            return '>20k'
        return f"{int(ok.iloc[0]['server_size']):,}"

    rows.append([
        f'{alpha:.1f}',
        f'{js_b:.4f}',
        min_srv('red_eu'),
        min_srv('red_pd'),
        min_srv('red_pdw'),
    ])

_box_table(headers, rows,
           title=f'Минимальный server_size для >{threshold:.0f}% JS reduction')

## 10. Детальный разбор одного случая: α=0.3, server=3000 vs 15000

In [ ]:
rng_detail = np.random.default_rng(0)

for srv_size in [3000, 15000]:
    per_class_srv   = srv_size // N_CLASSES
    per_class_avail = PER_CLASS_TOTAL - per_class_srv
    counts   = dirichlet_partition(per_class_avail, N_CLIENTS, 0.3, N_CLASSES, rng_detail)
    profiles = build_profiles(counts)
    Ps = [p['P'] for p in profiles]

    print(f'\n{'='*70}')
    print(f'  α=0.3, server_size={srv_size}  ({srv_size/TRAIN_SIZE*100:.0f}% train)')
    print(f'  Сэмплов клиентов : {[p["n"] for p in profiles]}')
    print(f'  JS до            : {mean_pairwise_js(Ps):.4f}')
    print(f'{'='*70}')

    for name, fn in [('exact-uniform', algo_exact_uniform),
                     ('prop-deficit',  algo_prop_deficit),
                     ('prop-def+w',    algo_prop_deficit_weighted)]:
        sched = fn(profiles, per_class_srv, N_CLASSES)
        Qs    = effective_distributions(profiles, sched)
        js_after = mean_pairwise_js(Qs)
        reduction = (mean_pairwise_js(Ps) - js_after) / mean_pairwise_js(Ps) * 100
        total_per_client = sched.sum(axis=1)
        budget_used_pct  = sched.sum() / (per_class_srv * N_CLASSES) * 100

        print(f'  [{name:20s}]  JS после = {js_after:.4f}  ({reduction:+.1f}%)  '
              f'бюджет={budget_used_pct:.0f}%  '
              f'per client={[int(x) for x in total_per_client]}')
    rng_detail = np.random.default_rng(0)  # reset for consistent comparison

## 11. Выводы

*(Заполни после запуска — ячейка для записи результатов)*

In [ ]:
print('═'*70)
print('  ИТОГОВЫЕ ВЫВОДЫ')
print('═'*70)

# Для каждого alpha: какой алгоритм лучше при server=3000?
srv = 3000
print(f'\nПри server_size={srv}:')
print(f'{"alpha":<8} {"JS_before":<12} {"exact-uni":<12} {"prop-def":<12} {"prop-def+w":<12} {"winner"}')
print('-'*65)
for alpha in ALPHAS:
    r = df[(df['alpha'] == alpha) & (df['server_size'] == srv)].iloc[0]
    js_b = r['js_before']
    eu   = r['red_eu']
    pd   = r['red_pd']
    pdw  = r['red_pdw']
    best_val = max(eu, pd, pdw)
    winner = 'eu' if eu == best_val else ('pd' if pd == best_val else 'pdw')
    print(f'{alpha:<8.1f} {js_b:<12.4f} {eu:<12.1f} {pd:<12.1f} {pdw:<12.1f} ← {winner}')

print(f'\nПри server_size=15000:')
srv2 = 15000
print(f'{"alpha":<8} {"JS_before":<12} {"exact-uni":<12} {"prop-def":<12} {"prop-def+w":<12} {"winner"}')
print('-'*65)
for alpha in ALPHAS:
    r = df[(df['alpha'] == alpha) & (df['server_size'] == srv2)].iloc[0]
    js_b = r['js_before']
    eu   = r['red_eu']
    pd   = r['red_pd']
    pdw  = r['red_pdw']
    best_val = max(eu, pd, pdw)
    winner = 'eu' if eu == best_val else ('pd' if pd == best_val else 'pdw')
    print(f'{alpha:<8.1f} {js_b:<12.4f} {eu:<12.1f} {pd:<12.1f} {pdw:<12.1f} ← {winner}')